# Forecast To Slotting And Space

Slotting evidence is based on demand class, forecast volume, order pressure, and pallet-position delta.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Image, Markdown, display
except Exception:
    def display(value):
        print(value)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image(filename={self.filename!r})"

ROOT = Path.cwd()
if ROOT.name != "v7_rm_pm_forecast_planning":
    ROOT = Path("Ai miroservices/modeling/v7_rm_pm_forecast_planning").resolve()
OUT = ROOT / "outputs"
PLOTS = OUT / "plots"
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

def load_csv(name, **kwargs):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return pd.read_csv(path, **kwargs)

def load_json(name):
    path = OUT / name
    if not path.exists():
        raise FileNotFoundError(f"Missing artifact: {path}. Run PYTHONPATH=. python3 -m pipeline.run_all first.")
    return json.loads(path.read_text())

def show_plot(name):
    path = PLOTS / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        display(Markdown(f"Plot not generated: `{path}`"))

In [2]:
slotting = load_csv('slotting_readiness_v7.csv')
slotting.head(40)

,material_id,material_code,description,material_type,abc_class,fms_class,forecast_p50_sum,suggested_order_qty,current_pallet_positions,target_pallet_positions,pallet_positions_delta,accessibility_need,slotting_score,slotting_action,recommendation_status,rationale
0,5571d3cd-666f-4673-a41e-efeb313da005,100036,CAUSTIC SODA,raw_material,A,F,862563.22,877208.28,1000.00,2036353.05,2035353.05,highest_access,100.00,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
1,ed5d69d5-7370-4a93-b888-2aa711897187,101054,CALCIUM CARBONATE ( GROUND ),raw_material,A,F,577852.36,593967.78,71.43,97916.96,97845.53,highest_access,99.69,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
2,861bf563-9747-4508-96c0-5a0c976acbcd,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,raw_material,A,F,532574.66,546534.10,65.70,63146.22,63080.52,highest_access,99.31,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
3,44dd0de1-d770-4f74-bccb-e6fc16f574ca,100098,SORBITOL,raw_material,A,F,521717.77,537027.91,63.53,72859.23,72795.70,highest_access,99.13,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
4,3670333e-585b-4f10-9d15-912dcb65d820,100108,TALCUM POWDER,raw_material,A,F,248334.96,252851.54,50.00,29364.27,29314.27,highest_access,98.75,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
5,584db23d-9317-457f-9e7d-d7592e043210,101580,SODIUM SILICATE,raw_material,A,F,213317.08,215986.20,60.00,25180.12,25120.12,highest_access,98.44,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
6,2b39aadc-6592-406c-914f-482f4cbb7ab5,100323,BC COLOGNE BULK - IMPORTED,raw_material,A,F,189840.22,194099.94,50.00,22499.71,22449.71,highest_access,98.12,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
7,9a11c556-2240-4f35-ae53-f0c8c8a88fb4,100460,GALAXY LES 70,raw_material,A,F,177876.72,181439.40,50.00,21063.53,21013.53,highest_access,97.81,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
8,e77e97bd-5de0-41f8-afc3-037acf6b900b,100050,GLYCERINE,raw_material,A,F,111478.14,112380.12,56.00,13158.97,13102.97,highest_access,97.50,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."
9,decce9e4-20de-45c6-af1c-9d641542a43f,100714,COCOMIDOPROPYL BETAINE,raw_material,A,F,98509.75,100036.61,50.00,11665.18,11615.18,highest_access,97.19,prefer_forward_accessible_pick_face,HIGH_RISK_REVIEW,"v7 direct RM/PM forecast, ABC=A, FMS=F, horizo..."


In [3]:
slotting.groupby(['accessibility_need', 'slotting_action']).size().reset_index(name='materials')

,accessibility_need,slotting_action,materials
0,high_access,keep_or_standard_slot,1
1,high_access,prefer_forward_accessible_pick_face,214
2,high_access,release_or_downslot_excess_space,3
3,highest_access,prefer_forward_accessible_pick_face,17
4,standard_access,keep_or_standard_slot,34
5,standard_access,release_or_downslot_excess_space,19


In [4]:
slotting.sort_values("slotting_score", ascending=False)[
    ["material_code", "description", "abc_class", "fms_class", "forecast_p50_sum", "pallet_positions_delta", "accessibility_need", "slotting_score", "slotting_action"]
].head(50)

,material_code,description,abc_class,fms_class,forecast_p50_sum,pallet_positions_delta,accessibility_need,slotting_score,slotting_action
0,100036,CAUSTIC SODA,A,F,862563.22,2035353.05,highest_access,100.00,prefer_forward_accessible_pick_face
1,101054,CALCIUM CARBONATE ( GROUND ),A,F,577852.36,97845.53,highest_access,99.69,prefer_forward_accessible_pick_face
2,101293,FLUFF UNTREATED - GOLDEN ISLES G4881,A,F,532574.66,63080.52,highest_access,99.31,prefer_forward_accessible_pick_face
3,100098,SORBITOL,A,F,521717.77,72795.70,highest_access,99.13,prefer_forward_accessible_pick_face
4,100108,TALCUM POWDER,A,F,248334.96,29314.27,highest_access,98.75,prefer_forward_accessible_pick_face
5,101580,SODIUM SILICATE,A,F,213317.08,25120.12,highest_access,98.44,prefer_forward_accessible_pick_face
6,100323,BC COLOGNE BULK - IMPORTED,A,F,189840.22,22449.71,highest_access,98.12,prefer_forward_accessible_pick_face
7,100460,GALAXY LES 70,A,F,177876.72,21013.53,highest_access,97.81,prefer_forward_accessible_pick_face
8,100050,GLYCERINE,A,F,111478.14,13102.97,highest_access,97.50,prefer_forward_accessible_pick_face
9,100714,COCOMIDOPROPYL BETAINE,A,F,98509.75,11615.18,highest_access,97.19,prefer_forward_accessible_pick_face
